# E1 — Export Top-K

Dùng sau khi một lượt E1 đã train xong và `CELL 5: SUCCESS`.

Thành viên chỉ cần chỉnh 2 dòng ở Cell 1:
- `MODEL`
- `SEED`

Notebook sẽ kiểm tra checkpoint, gọi `src/inference/export_topk.py`, rồi xác minh output.


In [ ]:
# ==========================================
# CELL 1 — CONFIGURATION & ENVIRONMENT
# ==========================================

import sys
import torch
from pathlib import Path
from google.colab import drive

# ===== CHỈ CHỈNH 2 DÒNG =====
MODEL = "lightgcn"       # lightgcn / lightgcn_plus
SEED = 2024              # 2023 / 2024 / 2025

# ===== FIXED CONFIG =====
K = 20
VALID_MODELS = ["lightgcn", "lightgcn_plus"]
VALID_SEEDS = [2023, 2024, 2025]

assert MODEL in VALID_MODELS
assert SEED in VALID_SEEDS

print("=== ENVIRONMENT ===")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable GPU in Colab first."
print("GPU:", torch.cuda.get_device_name(0))

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/Data_SNA_Group14")
DATA_DIR = ROOT / "data/raw/amazon"
E1_DIR = ROOT / "experiments/E1" / MODEL / f"seed_{SEED}"
CHECKPOINT = E1_DIR / "checkpoints" / f"{MODEL}-amazon-{SEED}.pth"
METRICS = E1_DIR / "metrics.json"
EXPORT_DIR = ROOT / "experiments/predictions" / MODEL / f"seed_{SEED}"

assert DATA_DIR.is_dir(), f"Missing dataset: {DATA_DIR}"
assert CHECKPOINT.is_file(), f"Missing checkpoint: {CHECKPOINT}"
assert METRICS.is_file(), f"Missing metrics.json: {METRICS}"

print("\n=== EXPORT CONFIGURATION ===")
print("Model:", MODEL)
print("Seed:", SEED)
print("K:", K)
print("Checkpoint:", CHECKPOINT)
print("Export dir:", EXPORT_DIR)
print("\nCELL 1: SUCCESS")


In [ ]:
# ==========================================
# CELL 2 — SETUP REPOSITORIES
# ==========================================

import subprocess
from pathlib import Path

GROUP_REPO = Path("/content/project_sna_group14")
GROUP_REPO_URL = "https://github.com/trungkien1502/project_sna_group14.git"

RLMREC_REPO = Path("/content/RLMRec")
RLMREC_URL = "https://github.com/HKUDS/RLMRec.git"
RLMREC_COMMIT = "22413752246de3dee8ab0d509f7f7a8889080f95"

if GROUP_REPO.exists():
    subprocess.run(["git", "-C", str(GROUP_REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", GROUP_REPO_URL, str(GROUP_REPO)], check=True)

EXPORT_SCRIPT = GROUP_REPO / "src/inference/export_topk.py"
assert EXPORT_SCRIPT.is_file(), f"Missing exporter: {EXPORT_SCRIPT}"

if not RLMREC_REPO.exists():
    subprocess.run(["git", "clone", RLMREC_URL, str(RLMREC_REPO)], check=True)

origin = subprocess.check_output(
    ["git", "-C", str(RLMREC_REPO), "remote", "get-url", "origin"],
    text=True
).strip()
assert origin == RLMREC_URL, f"Unexpected RLMRec origin: {origin}"

current_commit = subprocess.check_output(
    ["git", "-C", str(RLMREC_REPO), "rev-parse", "HEAD"],
    text=True
).strip()

if current_commit != RLMREC_COMMIT:
    status = subprocess.check_output(
        ["git", "-C", str(RLMREC_REPO), "status", "--porcelain"],
        text=True
    ).strip()
    assert not status, "RLMRec has local changes; restart runtime or inspect them."
    subprocess.run(["git", "-C", str(RLMREC_REPO), "fetch", "origin"], check=True)
    subprocess.run(
        ["git", "-C", str(RLMREC_REPO), "checkout", "--detach", RLMREC_COMMIT],
        check=True
    )

target = RLMREC_REPO / "data" / "amazon"
target.parent.mkdir(parents=True, exist_ok=True)

if target.is_symlink():
    assert target.resolve() == DATA_DIR.resolve()
elif target.exists():
    raise RuntimeError(f"{target} exists and is not a symlink.")
else:
    target.symlink_to(DATA_DIR, target_is_directory=True)

print("Exporter:", EXPORT_SCRIPT)
print("RLMRec commit:", RLMREC_COMMIT)
print("Dataset:", target.resolve())
print("\nCELL 2: SUCCESS")


In [ ]:
# ==========================================
# CELL 3 — RUN TOP-K EXPORTER
# ==========================================

import subprocess
import sys

cmd = [
    sys.executable,
    str(EXPORT_SCRIPT),
    "--repo", str(RLMREC_REPO),
    "--root", str(ROOT),
    "--model", MODEL,
    "--seed", str(SEED),
    "--k", str(K),
]

print("Running:")
print(" ".join(cmd))
subprocess.run(cmd, check=True)

print("\nCELL 3: SUCCESS")


In [ ]:
# ==========================================
# CELL 4 — VERIFY EXPORT OUTPUT
# ==========================================

import json
import numpy as np

TOPK_FILE = EXPORT_DIR / "topk_test.npz"
META_FILE = EXPORT_DIR / "export_meta.json"
VERIFICATION_FILE = EXPORT_DIR / "export_verification.json"

for path in [TOPK_FILE, META_FILE, VERIFICATION_FILE]:
    print(path.name, ":", "FOUND" if path.is_file() else "MISSING")
    assert path.is_file(), f"Missing {path}"

verification = json.loads(VERIFICATION_FILE.read_text(encoding="utf-8"))
meta = json.loads(META_FILE.read_text(encoding="utf-8"))

assert verification.get("status") == "passed", verification

data = np.load(TOPK_FILE)
required = {"user_ids", "item_ids", "scores"}
assert required.issubset(set(data.files)), data.files

print("\n=== EXPORT VERIFICATION ===")
print("Status:", verification["status"])
print("Model:", meta.get("model"))
print("Seed:", meta.get("seed"))
print("Top-K:", meta.get("top_k"))
print("Test users:", meta.get("test_user_count"))
print("item_ids shape:", data["item_ids"].shape)
print("scores shape:", data["scores"].shape)

print("\n=== METRICS FROM EXPORTED TOP-K ===")
for name, value in verification.get("test_metrics_from_export", {}).items():
    print(f"{name}: {value:.6f}")

print("\nEXPORT COMPLETE: PASS")
print("Output:", EXPORT_DIR)
